In [ ]:
# --- Setup ---------------------------------------------------------------
import os, sys, time, json, contextlib, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyarrow.parquet as pq

LESSON_DIR = Path.cwd()
DATA_DIR = Path(os.environ.get("SASSAFRAS_DATA", LESSON_DIR / ".." / "data"))      # the two Zenodo parquet files
PREP_DIR = LESSON_DIR / "prepared"                                            # cache written by Part 0
PREP_DIR.mkdir(exist_ok=True)
FLARE_REPO = Path(os.environ.get("FLARE_REPO", LESSON_DIR.parent / ".." / "flare")) # clone of applecider-ml/flare
if FLARE_REPO.exists():
    sys.path.insert(0, str(FLARE_REPO))

import flare
from flare import data as D
from flare.features import extract_from_array
assert hasattr(flare, "load_classifier"), "wrong `flare` on the path: run this notebook from its own folder"
print("flare", flare.__version__, "from", flare.__file__)

HEADERS = DATA_DIR / "sassafrass_headers.parquet"
PHOT = DATA_DIR / "sassafrass_phot.parquet"
print("Zenodo files present:", HEADERS.exists() and PHOT.exists())

SEED = 42
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
TEACHING_CLASS = {
    "SNIa-SALT3": "SN Ia", "SNIa-91bg": "SN Ia", "SNIax": "SN Ia",
    "SNIb+HostXT_V19": "SN Ibc", "SNIc+HostXT_V19": "SN Ibc", "SNIcBL+HostXT_V19": "SN Ibc",
    "SNIIb+HostXT_V19": "SN II", "SNIIn+HostXT_V19": "SN II",
    "SLSN-I+host": "SLSN", "TDE": "TDE",
    "KN_B19": "KN", "KN_K17": "KN",
    "CART": "CART", "ILOT": "ILOT",
}
CLASSES = ["SN Ia", "SN Ibc", "SN II", "SLSN", "TDE", "KN", "CART", "ILOT"]
CLASS_ID = {c: i for i, c in enumerate(CLASSES)}
CLASS_COLOR = dict(zip(CLASSES, plt.cm.magma(np.linspace(0.08, 0.88, len(CLASSES)))))

HEADER_COLS = ["SNID", "obj_type", "telescope", "REDSHIFT_FINAL", "HOSTGAL_PHOTOZ", "HOSTGAL_SNSEP",
               "HOSTGAL_MAG_g", "HOSTGAL_MAG_r", "HOSTGAL_MAG_i", "PEAKMJD", "MJD_DETECT_FIRST",
               "NOBS", "MWEBV", "SIM_PEAKMAG_r", "SIM_MODEL_NAME"]
headers = pq.read_table(HEADERS, columns=HEADER_COLS).to_pandas(ignore_metadata=True)
headers["SNID"] = headers["SNID"].str.strip()
headers["cls"] = headers["obj_type"].map(TEACHING_CLASS)
print(f"{len(headers):,} simulated objects, {headers.SNID.nunique():,} unique IDs")
headers.head()

In [ ]:
census = headers.groupby(["cls", "obj_type"]).size().unstack(0).fillna(0).astype(int)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
counts = headers.cls.value_counts().reindex(CLASSES)
axes[0].bar(CLASSES, counts.values, color=[CLASS_COLOR[c] for c in CLASSES])
axes[0].set_ylabel("objects in SASSAFRAS"); axes[0].set_title("Class census (all three follow-up telescopes)")
axes[0].tick_params(axis="x", rotation=30)
for c in CLASSES:
    z = headers.loc[headers.cls == c, "REDSHIFT_FINAL"]
    axes[1].hist(z, bins=np.linspace(0, 1.05, 43), histtype="step", lw=1.6, color=CLASS_COLOR[c], label=c)
axes[1].set_xlabel("redshift"); axes[1].set_ylabel("objects"); axes[1].set_title("Where in the universe they sit")
axes[1].legend(fontsize=8, ncol=2)
plt.tight_layout(); plt.show()
print(headers.groupby("cls").REDSHIFT_FINAL.median().reindex(CLASSES).round(3).rename("median z").to_frame().T)